In [ ]:
#| default_exp prune.prune_callback

In [ ]:
#| include: false
from nbdev.showdoc import *

In [ ]:
#| export
from fastai.vision.all import *
from fastai.callback.all import *
from fasterai.prune.pruner import *
from fasterai.core.criteria import *
from fasterai.core.ratio import as_fraction
from fasterai.core.schedule import *

from collections import defaultdict
import torch
import torch.nn as nn

## Overview

The `PruneCallback` integrates structured pruning into the fastai training loop. Unlike sparsification (which zeros weights), pruning physically removes network structures (filters, channels) to reduce model size and computation.

**Key Differences from SparsifyCallback:**
- Removes structures entirely (not just zeros)
- Uses `torch-pruning` library for dependency handling
- Supports various pruning criteria and schedules

In [ ]:
#| export
class PruneCallback(Callback):
    """Prune the model during training, with `fasterai.prune.Pruner`.
    A prune replaces the parameters of every layer it shrinks, so the optimizer is re-pointed at the
    live ones after each one: the state (momentum, and the rest) of a replaced parameter is dropped,
    the state of a parameter that survived the prune is kept, and so are the groups, the hypers,
    fastai's no-weight-decay and force-train marks and the freeze — torch-pruning re-creates the
    parameters it replaces requiring grad, and a frozen group stays frozen."""
    def __init__(self,
                 pruning_ratio,  # Filters to remove, a fraction in [0, 1] (0.4 = 40%), or a per-layer dict
                 schedule,       # When to prune, from `fasterai.core.schedule` (e.g. one_shot, agp)
                 context,        # 'local' or 'global'; a dict of ratios needs 'local'
                 criteria,       # How to select filters to prune, from `fasterai.core.criteria`
                 *args,
                 **kwargs
    ):
        store_attr()
        self.sparsity_levels = []
        self._is_per_layer = False
        self.extra_args = args
        self.extra_kwargs = kwargs
        self._validate_pruning_ratio()

    def _build_pruning_schedule(self, sched_func):
        "Create a schedule function compatible with torch-pruning's Pruner"
        start_val, end_val = self.schedule.start_val, self.schedule.end_val
        def scheduler(pruning_ratio, steps, start=start_val, end=end_val):
            return [
                sched_func(start, end, i / float(steps)) * pruning_ratio
                for i in range(steps + 1)
            ]
        return scheduler

    def _validate_pruning_ratio(self) -> None:
        "Read pruning_ratio as a fraction, supporting both a single value and a per-layer dict"
        self._is_per_layer = isinstance(self.pruning_ratio, dict)
        if self._is_per_layer:
            if self.context != 'local':
                raise ValueError("Per-layer pruning_ratio dict requires context='local' "
                                 "(global context compares importance across layers).")
            self.pruning_ratio = {l: as_fraction(r, 'pruning_ratio', layer=l)
                                  for l, r in self.pruning_ratio.items()}
        else:
            self.pruning_ratio = as_fraction(self.pruning_ratio, 'pruning_ratio', allow_zero=False)

    def before_fit(self) -> None:
        "Setup pruner before training"
        n_batches_per_epoch = len(self.learn.dls.train)
        total_training_steps = n_batches_per_epoch * self.learn.n_epoch
        self._validate_pruning_ratio()

        self.example_inputs, _ = self.learn.dls.one_batch()

        pruning_schedule = self._build_pruning_schedule(self.schedule.sched_func)
        # nothing single to log for a per-layer dict: torch-pruning schedules each layer on its own
        self.sparsity_levels = [] if self._is_per_layer else pruning_schedule(self.pruning_ratio, total_training_steps)

        self.pruner = Pruner(
            self.learn.model,
            criteria=self.criteria,
            pruning_ratio=self.pruning_ratio, 
            context=self.context,
            iterative_steps=total_training_steps, 
            schedule=pruning_schedule,
            *self.extra_args, 
            **self.extra_kwargs
        )
        
    def before_step(self) -> None:
        "Apply pruning before optimizer step"
        if self.training: 
            self.pruner.prune_model()
            self._rebind_opt()

    def _rebind_opt(self) -> None:
        "Re-point `learn.opt` at the model's live parameters, keeping the state of those the prune spared"
        opt = self.learn.opt  # re-pointed in place: rebuilding it would lose this fit's state and hypers
        groups = L(self.learn.splitter(self.learn.model))
        opt.param_lists = L(L(g) for g in groups) if isinstance(groups[0], (L, list)) else L([groups])
        live = {id(p) for g in opt.param_lists for p in g}
        for holder in (opt, getattr(opt, 'opt', None)):  # fastai's optimizer, and the torch one a wrapper holds
            state = getattr(holder, 'state', None)
            if isinstance(state, dict):
                holder.state = defaultdict(dict, {p: s for p, s in state.items() if id(p) in live})
        if not self.learn.wd_bn_bias:
            for s in self.learn._bn_bias_state(True): s['do_wd'] = False
        if self.learn.train_bn:
            for s in self.learn._bn_bias_state(False): s['force_train'] = True
        if opt.frozen_idx: opt.freeze_to(opt.frozen_idx)  # torch-pruning re-creates its parameters unfrozen

    def after_epoch(self) -> None:
        "Log the pruning ratio reached after each epoch"
        if self._is_per_layer:
            print(f'Pruning {len(self.pruning_ratio)} layers to per-layer targets (epoch {self.epoch})')
            return
        completed_steps = (self.epoch + 1) * len(self.learn.dls.train)
        if completed_steps > 0 and completed_steps <= len(self.sparsity_levels):
            current_ratio = self.sparsity_levels[completed_steps - 1]
            print(f'Pruning ratio at the end of epoch {self.epoch}: {current_ratio:.2%}')

In [ ]:
show_doc(PruneCallback)

## Usage Example

```python
from fasterai.prune.prune_callback import PruneCallback
from fasterai.core.schedule import agp, one_shot
from fasterai.core.criteria import large_final

# Uniform: prune 30% of parameters using automated gradual pruning
cb = PruneCallback(
    pruning_ratio=0.3,       # Remove 30% of parameters
    schedule=agp,            # Gradual pruning (cubic decay)
    context='global',        # Prune globally across all layers
    criteria=large_final     # Keep weights with largest magnitude
)
learn.fit(10, cbs=[cb])

# Per-layer: prune each layer to a different ratio (requires context='local')
cb = PruneCallback(
    pruning_ratio={'layer1': 0.3, 'layer3': 0.6},
    schedule=one_shot,
    context='local',
    criteria=large_final
)
learn.fit(10, cbs=[cb])
```

---

## See Also

- [Pruner](pruner.html) - Core structured pruning class used by this callback
- [Schedules](../core/schedules.html) - Control pruning progression during training
- [Criteria](../core/criteria.html) - Importance measures for selecting filters to prune
- [SparsifyCallback](../sparse/sparsify_callback.html) - Unstructured pruning alternative

Tests live in `nbs/tests/test_prune_callback.ipynb`.